# R2-S2 step 1 - per-recording measurements for the training-partition EDA (Colab)

CPU runtime is enough. Run all cells top to bottom; it is safe to rerun after a disconnect (finished shards are skipped, a finished build is detected).

**Scope:** Protocol B **train** only - genuine, MeloTTS and VITS recordings of the 151 training speakers (the same 4,530 recordings and the same plan hash
as R2-S1). Validation, test and ChatterboxTTS audio is never read.

**Produces** in `data/eda/r2_s2_B_train/` on Drive:
1. `recording_stats_B_train_v1.parquet` - duration, RMS, peak, clipping, DC offset, leading/trailing silence, silence fraction, noise floor, high-band energy shares;
2. `mean_logmel_B_train_v1.npz` - the mean log-mel spectrum of every recording (for the average-spectrum figure);
3. `examples_B_train_v1.npz` - int16 waveforms of three paired example sentences (genuine, Melo, VITS) for the waveform / log-mel figure;
4. `r2_s2_manifest.json` - SHA-256 of every output, plan and split hashes, counts, checks;
5. `reports/r2/r2_s2_colab_<timestamp>.json` - the evidence report.

The figures themselves are built locally from these files (`python scripts/r2_s2_eda.py figures`), they need no audio.

Steps: check the synced code -> pre-flight -> unit tests -> copy the train shards -> measure -> independent cross-checks -> evidence.

In [ ]:
import os, sys, json, subprocess, time, hashlib, platform, shutil
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

ROOT = Path(os.environ.get("ECHOPROOF_ROOT", "/content/drive/MyDrive/PAI&AIES Project"))
assert ROOT.exists(), f"project folder not found: {ROOT}"

SOURCES = {"__init__eda.py": "src/echoproof/eda/__init__.py", "measure.py": "src/echoproof/eda/measure.py", "figures.py": "src/echoproof/eda/figures.py",
           "r2_s1.py": "src/echoproof/features/r2_s1.py", "stats.py": "src/echoproof/features/stats.py", "acoustic.py": "src/echoproof/features/acoustic.py",
           "cohort.py": "src/echoproof/data/cohort.py", "splits.py": "src/echoproof/data/splits.py", "chunks.py": "src/echoproof/data/chunks.py",
           "seeding.py": "src/echoproof/training/seeding.py", "contracts.py": "src/echoproof/contracts.py",
           "r2_s2_eda.py": "scripts/r2_s2_eda.py", "test_r2_s2_eda.py": "tests/test_r2_s2_eda.py", "test_r2_s1_features.py": "tests/test_r2_s1_features.py"}
missing = [k for k, rel in SOURCES.items() if not (ROOT / rel).exists()]
assert not missing, f"not found under {ROOT}: {missing} - sync these files to Drive and rerun"
SHA = {k: hashlib.sha256((ROOT / rel).read_bytes()).hexdigest() for k, rel in SOURCES.items()}
print(json.dumps({k: v[:12] for k, v in SHA.items()}, indent=2))

In [ ]:
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
os.environ["PYTHONPATH"] = str(ROOT / "src") + os.pathsep + os.environ.get("PYTHONPATH", "")   # spawned workers import echoproof too
sys.dont_write_bytecode = True
sys.path.insert(0, str(ROOT / "src"))

import numpy, scipy, librosa, pyarrow, pandas
ENV = {"python": platform.python_version(), "numpy": numpy.__version__, "scipy": scipy.__version__, "librosa": librosa.__version__,
       "pyarrow": pyarrow.__version__, "pandas": pandas.__version__, "platform": platform.platform(), "cpu_count": os.cpu_count()}
print(json.dumps(ENV, indent=2))

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

INPUTS = {"cohort": ROOT / "data/manifests/cohort_v1.parquet", "split": ROOT / "data/splits/protocol_B_v1.parquet",
          "chunks": ROOT / "data/processed/chunks_v1.parquet", "split_report": ROOT / "reports/r1/split_report.json"}
for name, path in INPUTS.items():
    assert path.exists(), f"missing locked input {name}: {path}"

EXPECTED_COHORT_FILE_SHA256 = "51af27a2dd5c0577bf9c232287ac73b2f66af472ce3308721af83b2e48e8e816"
INPUT_SHA = {name: sha256_file(path) for name, path in INPUTS.items() if name != "split_report"}
print({k: v[:12] for k, v in INPUT_SHA.items()})
assert INPUT_SHA["cohort"] == EXPECTED_COHORT_FILE_SHA256, "the cohort manifest on Drive is not the frozen GR1 file - stop and compare it with the local copy"

import pyarrow.parquet as pq
FROZEN_B = json.loads(INPUTS["split_report"].read_text())["protocols"]["B"]["split_sha256"]
SPLIT_ROWS = [json.loads(r["payload_json"]) for r in pq.read_table(INPUTS["split"]).to_pylist()]
assert {r["split_hash"] for r in SPLIT_ROWS} == {FROZEN_B}, "protocol B split hash differs from the frozen value"
print("protocol B split hash matches the frozen value:", FROZEN_B[:12])

## 1. Unit tests on this runtime

In [ ]:
proc = subprocess.run([sys.executable, "-m", "pytest", "tests/test_r2_s2_eda.py", "-q", "-p", "no:cacheprovider"],
                      cwd=ROOT, env={**os.environ, "PYTHONPATH": str(ROOT / "src")}, capture_output=True, text=True)
print(proc.stdout[-3000:] + proc.stderr[-1500:])
UNIT_TESTS_PASSED = proc.returncode == 0
PYTEST_SUMMARY = next((l.strip() for l in reversed(proc.stdout.splitlines()) if " passed" in l or " failed" in l or " error" in l), "no summary line")
print("unit tests:", "PASSED" if UNIT_TESTS_PASSED else "FAILED", "|", PYTEST_SUMMARY)
assert UNIT_TESTS_PASSED, "do not build on a runtime where the tests fail"

## 2. Copy the shards that hold B-train recordings from Drive to local disk

In [ ]:
COHORT_ROWS = pq.read_table(INPUTS["cohort"], columns=["recording_id", "canonical_path", "retained", "component", "duration_s", "n_samples", "pcm_sha256"]).to_pylist()
BY_ID = {r["recording_id"]: r for r in COHORT_ROWS}
TRAIN_IDS = [r["recording_id"] for r in SPLIT_ROWS if r["partition"] == "train"]
needed = sorted({BY_ID[i]["canonical_path"].split("::")[0] for i in TRAIN_IDS})
SHARD_SRC, SHARD_DIR = ROOT / "data/processed/cohort_v1", Path("/content/shards")
SHARD_DIR.mkdir(parents=True, exist_ok=True)
print(len(TRAIN_IDS), "B-train recordings in", len(needed), "shards:", needed)
t0 = time.perf_counter()
for name in needed:
    src, dst = SHARD_SRC / name, SHARD_DIR / name
    assert src.exists(), f"shard missing on Drive: {src}"
    if dst.exists() and dst.stat().st_size == src.stat().st_size:
        continue
    for attempt in range(4):
        try:
            tmp = dst.with_name(dst.name + ".part")
            shutil.copyfile(src, tmp)
            assert tmp.stat().st_size == src.stat().st_size, "size differs after copy"
            os.replace(tmp, dst)
            break
        except OSError as exc:
            print(f"  {name}: attempt {attempt + 1} failed ({exc}); retrying")
            time.sleep(5 * 2 ** attempt)
    else:
        raise RuntimeError(f"could not copy {name}")
    print(f"  copied {name} ({dst.stat().st_size / 1e6:.0f} MB)")
COPY_SECONDS = round(time.perf_counter() - t0, 1)
print("shards ready in", SHARD_DIR, "|", COPY_SECONDS, "s")

## 3. Measure (progress is printed per shard)

In [ ]:
from echoproof.eda import measure

OUT_DIR = ROOT / "data/eda/r2_s2_B_train"
BUILD_SECONDS = None
if (OUT_DIR / measure.MANIFEST_NAME).exists():
    print("already built - loading the existing manifest")
    MANIFEST = json.loads((OUT_DIR / measure.MANIFEST_NAME).read_text())
else:
    t0 = time.perf_counter()
    MANIFEST = measure.build(ROOT, SHARD_DIR, OUT_DIR, seed=42, workers=os.cpu_count() or 1, resume=True, progress=print)
    BUILD_SECONDS = round(time.perf_counter() - t0, 1)
print(json.dumps({k: MANIFEST[k] for k in ("plan_sha256", "counts", "examples", "seconds_total")}, indent=2))

## 4. Independent cross-checks of the files on Drive

In [ ]:
import numpy as np, pandas as pd
from collections import Counter

problems = []
OUTPUT_HASHES = {}
for key, info in MANIFEST["outputs"].items():
    actual = sha256_file(OUT_DIR / info["file"])
    OUTPUT_HASHES[key] = actual
    if actual != info["sha256"]:
        problems.append(f"{key}: file hash differs from the manifest")

frame = pd.read_parquet(OUT_DIR / measure.OUTPUT_NAMES["stats"])
mel = np.load(OUT_DIR / measure.OUTPUT_NAMES["mel_mean"], allow_pickle=False)
ex = np.load(OUT_DIR / measure.OUTPUT_NAMES["examples"], allow_pickle=False)

# scope, re-derived from the split file (not from the build's plan)
split_train = set(TRAIN_IDS)
test_ids = {r["recording_id"] for r in SPLIT_ROWS if r["partition"] == "test"}
validation_ids = {r["recording_id"] for r in SPLIT_ROWS if r["partition"] == "validation"}
ids = set(frame["recording_id"])
if ids != split_train:
    problems.append("measured recordings differ from the B-train assignments")
if ids & (test_ids | validation_ids):
    problems.append("validation or test recordings appear in the measurements")
if any(":chatterbox:" in i or ":xtts:" in i for i in ids):
    problems.append("Chatterbox or XTTS recordings appear in the measurements")
if set(frame["component"]) != {"genuine", "melo", "vits"}:
    problems.append(f"components {sorted(set(frame['component']))}")
if mel["recording_id"].tolist() != frame["recording_id"].tolist():
    problems.append("mean log-mel rows are not aligned with the table")

# measurements against the manifest
dur_gap = max(abs(r.duration_s - BY_ID[r.recording_id]["duration_s"]) for r in frame.itertuples())
len_bad = [r.recording_id for r in frame.itertuples() if r.n_samples != BY_ID[r.recording_id]["n_samples"]]
if dur_gap > 1e-3 or len_bad:
    problems.append(f"durations/lengths differ from the cohort manifest (max gap {dur_gap}, {len(len_bad)} lengths)")
num = frame[list(measure.SCALAR_COLUMNS)].to_numpy(float)
if not np.isfinite(num).all():
    problems.append("non-finite measurements")

# example waveforms: sample-exact against the manifest hash
from echoproof.data import cohort
example_hash_bad = []
for stem in ex["stems"].tolist():
    for comp in ex["components"].tolist():
        rid = f"phonemedf:{comp}:{stem}"
        if rid not in split_train:
            problems.append(f"example {rid} is not a B-train recording")
        elif cohort.pcm_sha256(ex[f"wave__{stem}__{comp}"]) != BY_ID[rid]["pcm_sha256"]:
            example_hash_bad.append(rid)
if example_hash_bad:
    problems.append(f"example waveforms differ from the manifest hash: {example_hash_bad}")

SUMMARY = {"recordings": int(len(frame)), "by_component": dict(Counter(frame["component"])), "speakers": int(frame["speaker_id"].nunique()),
           "examples": ex["stems"].tolist(),
           "median_duration_s": {c: float(frame.loc[frame["component"] == c, "duration_s"].median()) for c in ("genuine", "melo", "vits")},
           "median_rms_dbfs": {c: float(frame.loc[frame["component"] == c, "rms_dbfs"].median()) for c in ("genuine", "melo", "vits")},
           "recordings_with_clipping": int((frame["clip_fraction"] > 0).sum()), "max_duration_gap_to_manifest_s": float(dur_gap)}
print(json.dumps(SUMMARY, indent=2))
print("problems:", problems or "none")

## 5. Verdict and evidence

In [ ]:
checks = {"unit_tests_passed": UNIT_TESTS_PASSED, "build_verification_passed": MANIFEST["verification"]["passed"],
          "spot_check_mismatches": MANIFEST["spot_check"]["mismatches"],
          "cohort_manifest_is_frozen_file": INPUT_SHA["cohort"] == EXPECTED_COHORT_FILE_SHA256,
          "split_hash_is_frozen_value": MANIFEST["inputs"]["split_sha256"] == FROZEN_B == MANIFEST["inputs"]["split_sha256_frozen_in_report"],
          "cross_check_problems": problems}
verified = (checks["unit_tests_passed"] and checks["build_verification_passed"] and not checks["spot_check_mismatches"]
            and checks["cohort_manifest_is_frozen_file"] and checks["split_hash_is_frozen_value"] and not problems)
evidence = {"schema_version": 1, "task": "R2-S2", "step": "measure", "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            "environment": ENV, "source_sha256": SHA, "input_sha256": INPUT_SHA, "output_sha256": OUTPUT_HASHES,
            "verdict": {"verified": bool(verified), "pytest_summary": PYTEST_SUMMARY, **checks}, "summary": SUMMARY, "build_manifest": MANIFEST,
            "timing": {"shard_copy_seconds": COPY_SECONDS, "build_seconds_this_session": BUILD_SECONDS, "build_seconds_recorded": MANIFEST["seconds_total"]}}
out_dir = ROOT / "reports" / "r2"
out_dir.mkdir(parents=True, exist_ok=True)
path = out_dir / f"r2_s2_colab_{time.strftime('%Y%m%dT%H%M%SZ', time.gmtime())}.json"
path.write_text(json.dumps(evidence, indent=2, sort_keys=True, default=str), encoding="utf-8")
print(json.dumps(evidence["verdict"], indent=2, default=str))
print("\nevidence written to:", path)